# bob, explained - Episode 11: The whole flow, by hand and by tool

The whole flow, by hand and by tool

Run `!pip install manim` and `from manim import *` once first, then this cell. Start at `-ql`; the class names listed at the top of the cell render a single section.


In [ ]:
%%manim -qm Ep11Flow
# =============================================================================
#  bob, explained - EPISODE 11: The whole flow, by hand and by tool
#  The whole flow, by hand and by tool
#
#  GENERATED by docs/manim/build.py from docs/manim/parts/. Do not edit here.
#
#  Prerequisite (once per notebook, in a cell of its own):
#      !pip install manim
#      from manim import *
#
#  Quality on the magic line above:  -ql draft   -qm medium   -qh 1080p60
#
#  Render one section instead of the whole episode by putting any of these
#  class names on the magic line:
#      E11S1Byhand
#      E11S2Bytool
#      E11S3Layers
#      E11S4Board
#      E11S5Files
# =============================================================================

# =============================================================================
#  shared prelude - palette, helpers and the BobScene base class.
#  docs/manim/build.py pastes this into the top of every episode cell.
# =============================================================================

from manim import *
import numpy as np

# ---------------------------------------------------------------- palette ----
BG    = "#11121a"
INK   = "#e8e8ea"
DIM   = "#8b93a7"
C_PY  = "#7aa2f7"   # blue    - Python / tools / the device description
C_VPR = "#f7768e"   # red     - VPR / external tools
C_RTL = "#9ece6a"   # green   - hardware, Verilog, things on the die
C_BIT = "#e0af68"   # amber   - configuration bits, FASM, the bitstream
C_GRF = "#bb9af7"   # purple  - graphs, JTAG, protocol
C_ERR = "#ff7a93"   # pink    - bugs, refusals, errors
MONO  = "monospace"


# ---------------------------------------------------------------- helpers ----
def mono(s, size=22, color=INK):
    """One line of monospace text (Pango crashes on '', so blanks become ' ')."""
    return Text(s if s else " ", font=MONO, font_size=size, color=color)


def code_block(lines, size=20, color=INK):
    g = VGroup(*[mono(l, size, color) for l in lines])
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.14)
    return g


def panel(mob, color=DIM, pad=0.32, fill=0.06):
    r = SurroundingRectangle(mob, color=color, buff=pad)
    r.set_fill(color, opacity=fill)
    return VGroup(r, mob)


def chip(label, color, w=2.6, h=0.95, size=22, weight="NORMAL"):
    box = RoundedRectangle(width=w, height=h, corner_radius=0.14,
                           color=color, stroke_width=3)
    box.set_fill(color, opacity=0.12)
    txt = Text(label, font_size=size, color=INK, weight=weight, line_spacing=0.75)
    if txt.width > w - 0.3:
        txt.scale_to_fit_width(w - 0.3)
    if txt.height > h - 0.2:
        txt.scale_to_fit_height(h - 0.2)
    return VGroup(box, txt.move_to(box.get_center()))


def arrow(a, b, color=DIM, buff=0.15, sw=3):
    return Arrow(a, b, buff=buff, color=color, stroke_width=sw,
                 max_tip_length_to_length_ratio=0.18)


def mux_symbol(color=C_RTL, h=1.9, w=0.8):
    """Classic trapezoid multiplexer symbol."""
    p = Polygon([-w / 2,  h / 2, 0], [w / 2,  h / 2 - 0.3, 0],
                [ w / 2, -h / 2 + 0.3, 0], [-w / 2, -h / 2, 0],
                color=color, stroke_width=3)
    p.set_fill(color, opacity=0.14)
    return p


def bitcells(n, size=0.3, on=(), color=C_BIT, off_color=DIM):
    """A strip of n little squares; indices in `on` are filled."""
    g = VGroup()
    for i in range(n):
        s = Square(size, color=off_color, stroke_width=1.6)
        if i in on:
            s.set_stroke(color).set_fill(color, opacity=0.85)
        g.add(s)
    g.arrange(RIGHT, buff=0.035)
    return g


def fieldbar(fields, total_w=11.0, h=0.62, size=15):
    """
    fields: [(label, nbits, color), ...] -> one horizontal bar split to scale,
    each slice labelled above and its bit range below. Returns VGroup(bar, labels, ranges).
    """
    nbits = sum(f[1] for f in fields)
    bar, labs, rngs = VGroup(), VGroup(), VGroup()
    x, lo = -total_w / 2, 0
    for label, n, col in fields:
        w = max(total_w * n / nbits, 0.34)
        r = Rectangle(width=w, height=h, color=col, stroke_width=2)
        r.set_fill(col, opacity=0.28).move_to(np.array([x + w / 2, 0, 0]))
        bar.add(r)
        t = Text(label, font_size=size, color=col)
        if t.width > w * 1.9:
            t.scale_to_fit_width(max(w * 1.9, 0.5))
        t.next_to(r, UP, buff=0.14)
        labs.add(t)
        rt = mono(f"{lo}" if n == 1 else f"{lo}..{lo + n - 1}", size - 2, DIM)
        rt.next_to(r, DOWN, buff=0.12)
        if rt.width > w * 1.9:
            rt.scale_to_fit_width(max(w * 1.9, 0.5))
        rngs.add(rt)
        x += w
        lo += n
    return VGroup(bar, labs, rngs)


def filecard(path, role, color):
    """A small card naming a repo file and what it is."""
    t = mono(path, 17, color)
    r = Text(role, font_size=14, color=DIM)
    g = VGroup(t, r).arrange(DOWN, aligned_edge=LEFT, buff=0.08)
    box = SurroundingRectangle(g, color=color, buff=0.16)
    box.set_fill(color, opacity=0.07)
    return VGroup(box, g)


def mid(a, b):
    """midpoint, defined here so nothing depends on manim exporting space_ops."""
    return (a + b) / 2


def clear_all(sc, run_time=0.6):
    if sc.mobjects:
        sc.play(*[FadeOut(m) for m in sc.mobjects], run_time=run_time)


class BobScene(Scene):
    def setup(self):
        self.camera.background_color = BG

    def heading(self, text, kicker=None):
        t = Text(text, font_size=32, color=INK, weight="BOLD")
        t.to_corner(UL).shift(DOWN * 0.1)
        rule = Line(LEFT * 6.6, RIGHT * 6.6, color=DIM, stroke_width=1.5)
        rule.next_to(t, DOWN, buff=0.2).align_to(t, LEFT)
        g = VGroup(t, rule)
        self.play(FadeIn(t, shift=RIGHT * 0.3), Create(rule), run_time=0.7)
        if kicker:
            k = Text(kicker, font_size=19, color=DIM)
            if k.width > 13.0:
                k.scale_to_fit_width(13.0)
            k.next_to(rule, DOWN, buff=0.16).align_to(t, LEFT)
            g.add(k)
            self.play(FadeIn(k), run_time=0.4)
        return g

    def titlecard(self, number, title, subtitle):
        n = Text(number, font_size=26, color=C_BIT, weight="BOLD")
        t = Text(title, font_size=60, color=INK, weight="BOLD")
        s = Text(subtitle, font_size=26, color=DIM)
        if t.width > 12.5:
            t.scale_to_fit_width(12.5)
        if s.width > 12.5:
            s.scale_to_fit_width(12.5)
        g = VGroup(n, t, s).arrange(DOWN, buff=0.4)
        self.play(FadeIn(n), run_time=0.4)
        self.play(Write(t), run_time=1.1)
        self.play(FadeIn(s, shift=UP * 0.2), run_time=0.7)
        self.wait(1.6)
        self.play(FadeOut(g), run_time=0.6)

    def files_used(self, inputs, generated, verified):
        """Closing card: what this episode's topic is built from and checked by."""
        self.heading("Files", "what this part is written in, what is generated, and what proves it")
        cols = []
        for title, items, col in (("written by hand", inputs, C_RTL),
                                  ("generated", generated, C_PY),
                                  ("verified by", verified, C_BIT)):
            head = Text(title, font_size=21, color=col, weight="BOLD")
            cards = VGroup(*[filecard(p, r, col) for p, r in items])
            cards.arrange(DOWN, aligned_edge=LEFT, buff=0.18)
            g = VGroup(head, cards).arrange(DOWN, aligned_edge=LEFT, buff=0.28)
            cols.append(g)
        row = VGroup(*cols).arrange(RIGHT, buff=0.7, aligned_edge=UP)
        if row.width > 13.2:
            row.scale_to_fit_width(13.2)
        row.next_to(self.mobjects[1], DOWN, buff=0.55).set_x(0)
        for c in cols:
            self.play(FadeIn(c, shift=UP * 0.2), run_time=0.7)
        self.wait(2.4)

# =============================================================================
#  EPISODE 11 - The whole flow: by hand, and by tool
# =============================================================================

def s1_byhand(sc):
    sc.heading("Before there were tools, there was an API",
               "host/bitstream.py - place a LUT, name its inputs, let a BFS router connect them")

    code = code_block([
        "from bitstream import Design, LUT",
        "",
        "d = Design()",
        "a, b = d.input(0), d.input(1)",
        "g = d.lut(2, 3, LUT.and2(), [a, b])     # an AND at tile (2,3)",
        "d.output(0, g)                          # to LD0",
        "bs = d.build()                          # place, route, pack 18560 bits",
    ], 21, INK)
    cp = panel(code, C_PY)
    cp.shift(UP * 1.5)
    sc.play(FadeIn(cp), run_time=1.0)

    how = code_block([
        "build() runs a breadth-first search over the routing graph, allocating one",
        "mux per hop. A mux already carrying the same signal is free to traverse",
        "again - which is how fanout works with no special case at all.",
        "",
        "It still exists, and hwtest still uses it: the hand-built designs are the",
        "regression that every board run starts with.",
    ], 19, INK)
    how[4].set_color(C_BIT); how[5].set_color(C_BIT)
    how.next_to(cp, DOWN, buff=0.8).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in how], lag_ratio=0.18), run_time=2.0)

    note = Text("This is the honest way to learn a fabric: if you can place and route it "
                "by hand, you understand it.", font_size=20, color=C_BIT)
    note.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.35)
    sc.play(FadeIn(note), run_time=0.9)
    sc.wait(2.0)


def s2_bytool(sc):
    sc.heading("./bob build", "seven steps, and every one of them is checked before the next runs")

    steps = [
        ("synthesis", "yosys onto bob cells", C_VPR),
        ("equivalence", "source == netlist == golden, 300 cycles", C_ERR),
        ("place and route", "VPR on the committed graph, or bob's own PnR", C_VPR),
        ("FASM", "features, legality-checked against device.json", C_BIT),
        ("bits", "bitgen; the ctrl tile gets the clock setting", C_BIT),
        ("model", "model.py with these bits == the source trace", C_ERR),
        (".bit", "chain + BRAM sections + META + file CRC", C_PY),
    ]
    g = VGroup()
    for i, (a, b, col) in enumerate(steps):
        n = mono(f"{i + 1}", 20, col)
        t = mono(a, 20, col)
        w = Text(b, font_size=17, color=DIM)
        g.add(VGroup(n, t, w).arrange(RIGHT, buff=0.4, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 0.55)
        r[2].align_to(g[0][2], LEFT).shift(RIGHT * 3.4)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.3)
    if g.width > 12.6:
        g.scale_to_fit_width(12.6)
    g.next_to(sc.mobjects[1], DOWN, buff=0.7).set_x(0)
    for r in g:
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.4)

    two = Text("Two commands, end to end:   ./bob build design.v    then    ./bob load x.bit",
               font_size=22, color=C_BIT)
    two.scale_to_fit_width(12.8).to_edge(DOWN, buff=0.4)
    sc.play(FadeIn(two), run_time=0.8)
    sc.wait(2.0)


def s3_layers(sc):
    sc.heading("Why you can believe the result",
               "each layer is checked against something that was NOT derived from it")

    layers = [
        ("references", "UG470 / 473 / 474 / 479, OpenFPGA, VPR", C_GRF),
        ("Python models", "model.py, packets.Controller, chainbits CRC", C_PY),
        ("RTL testbenches", "expectations come from the models, never from the RTL", C_RTL),
        ("mutation tests", "break each guard on purpose - a test MUST fail", C_ERR),
        ("golden co-simulation", "source || golden netlist || the whole FPGA from a .bit", C_BIT),
        ("the stand-in board", "every hardware check, on a good board and a broken one", C_VPR),
        ("the PYNQ-Z2", "make hwtest M=Mx, appended to results.log", INK),
    ]
    g = VGroup()
    for a, b, col in layers:
        g.add(VGroup(mono(a, 20, col), Text(b, font_size=17, color=DIM))
              .arrange(RIGHT, buff=0.5, aligned_edge=DOWN))
    for r in g:
        r[1].align_to(g[0][1], LEFT).shift(RIGHT * 4.4)
    g.arrange(DOWN, aligned_edge=LEFT, buff=0.3)
    if g.width > 13.0:
        g.scale_to_fit_width(13.0)
    g.next_to(sc.mobjects[1], DOWN, buff=0.6).set_x(0)
    for i, r in enumerate(g):
        sc.play(FadeIn(r, shift=RIGHT * 0.2), run_time=0.4)
        if i:
            sc.play(GrowArrow(arrow(g[i - 1].get_left() + LEFT * 0.25,
                                    r.get_left() + LEFT * 0.25, DIM, 0.02, 2)),
                    run_time=0.12)

    counts = Text("28 744 testbench checks  ·  193 pytest tests  ·  59 mutants, all killed  ·  "
                  "57 logged board runs", font_size=20, color=C_BIT)
    counts.scale_to_fit_width(13.0).to_edge(DOWN, buff=0.3)
    sc.play(FadeIn(counts), run_time=0.9)
    sc.wait(2.2)


def s4_board(sc):
    sc.heading("What a board run actually does",
               "make hwtest M=M16 - the regression first, so a new milestone cannot break an old one")

    g = VGroup(
        chip("idcode", C_GRF, 2.2, 0.7, 19),
        chip("bypass", C_GRF, 2.2, 0.7, 19),
        chip("selftest", C_GRF, 2.2, 0.7, 19),
    ).arrange(RIGHT, buff=0.4).shift(UP * 2.0)
    sc.play(FadeIn(g), run_time=0.6)
    gl = Text("the regression - every run, in this order", font_size=18, color=DIM)
    gl.next_to(g, DOWN, buff=0.22)
    sc.play(FadeIn(gl), run_time=0.4)

    more = code_block([
        "then the milestone's own checks, for example on M16:",
        "",
        "   the full M15 list      chain, frames, CRC, GTS, GSR/GWE, CAPTURE,",
        "                          BRAM modes, DSP, partial reconfiguration",
        "   bob-big / pnr-big      a 56-CLB design that could not fit the old grid,",
        "                          through VPR and through bob's own PnR",
        "   live checks            a person at the switches, with goals and a timer",
    ], 19, INK)
    more[0].set_color(DIM)
    more.next_to(gl, DOWN, buff=0.6).set_x(0)
    sc.play(LaggedStart(*[FadeIn(l) for l in more], lag_ratio=0.15), run_time=2.0)

    rule = code_block([
        "Two rules that make these trustworthy:",
        "   every check runs against the STAND-IN board first, passing and failing",
        "   every run is appended to docs/hwtest/results.log and never edited",
    ], 19, C_BIT)
    rule[0].set_color(DIM)
    rule.next_to(more, DOWN, buff=0.6).set_x(0)
    sc.play(FadeIn(rule), run_time=0.9)
    sc.wait(2.2)


def s5_files(sc):
    sc.files_used(
        inputs=[("host/bitstream.py", "the hand-design API and its BFS router"),
                ("tools/bob/cli.py", "./bob build and load"),
                ("host/hwtest.py", "15 milestone check lists")],
        generated=[("build/bit/*.bit", "loadable bitstreams"),
                   ("build/cosim/", "the co-simulation harness"),
                   ("docs/hwtest/results.log", "57 board runs, append only")],
        verified=[("sim/run_cosim_sim.sh", "5822 checks across 22 design builds"),
                  ("tests/test_hwtest_fake.py", "the stand-in board"),
                  ("the PYNQ-Z2", "M0-M15 all passed; M16 pending")])


EP11 = [s1_byhand, s2_bytool, s3_layers, s4_board, s5_files]


class Ep11Flow(BobScene):
    def construct(self):
        self.titlecard("EPISODE 11", "The whole flow",
                       "by hand, by tool, and why you can believe the answer")
        for i, part in enumerate(EP11):
            part(self)
            if i < len(EP11) - 1:
                clear_all(self)


class E11S1Byhand(BobScene):
    def construct(self): s1_byhand(self)


class E11S2Bytool(BobScene):
    def construct(self): s2_bytool(self)


class E11S3Layers(BobScene):
    def construct(self): s3_layers(self)


class E11S4Board(BobScene):
    def construct(self): s4_board(self)


class E11S5Files(BobScene):
    def construct(self): s5_files(self)